# HTDemucs — Fine-Tune on binauralMUSDB18

Fine-tunes the full pre-trained **HTDemucs** backbone on the binauralMUSDB18-HQ dataset  
using the same **multi-domain loss** that HTDemucs was originally trained with:

$$\mathcal{L} = \lambda_{\text{td}} \cdot \underbrace{\frac{1}{S}\sum_s \|\hat{s} - s\|_1}_{\text{time-domain L1}} + \lambda_{\text{spec}} \cdot \underbrace{\frac{1}{S}\sum_s \sum_{n} \|\,|\text{STFT}_n(\hat{s})| - |\text{STFT}_n(s)|\,\|_1}_{\text{multi-scale spectrogram L1}}$$

**Memory strategy** (HTDemucs ~80M params needs careful handling):  
- `torch.cuda.amp` (FP16) — halves activation memory  
- `ACCUM_STEPS` gradient accumulation — effective batch = `BATCH_SIZE × ACCUM_STEPS`  
- `BATCH_SIZE = 1` by default; increase only if VRAM allows  

> **Prerequisite:** `pip install demucs`

In [1]:
import sys
from pathlib import Path

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import math
import time
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, random_split

import torchaudio
import numpy as np
import matplotlib.pyplot as plt

from demucs.pretrained import get_model
from msslnet.dataset import MusdbSpatialDataset

## Configuration

In [2]:
# ── Remote or local ───────────────────────────────────────────────────────────
remote   = True
training = True

# ── Paths ─────────────────────────────────────────────────────────────────────
if remote:
    DATASET_ROOT = Path("/nas/home/macerbi/Dataset/binauralMUSDB18HQ")
    CKPT_DIR     = Path("/nas/home/macerbi/msslnet/runs/htdemucs_finetune")
else:
    DATASET_ROOT = Path(r"D:\Polimi\PhD\Dataset\binauralMUSDB18HQ")
    CKPT_DIR     = Path(r"H:\Il mio Drive\Polimi\PhD\Progetto di Ricerca\MSS & SSL\msslnet\runs\htdemucs_finetune")

CKPT_DIR.mkdir(parents=True, exist_ok=True)

if training:
    _run_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
else:
    _run_ts = "YYYYMMDD_HHMMSS"   # <-- set to the run you want to resume/evaluate
CKPT_PATH = CKPT_DIR / f"htdemucs_{_run_ts}.pt"

# ── Training hyper-parameters ─────────────────────────────────────────────────
N_EPOCHS    = 100
BATCH_SIZE  = 1    # increase to 2 only if VRAM > ~20 GB
ACCUM_STEPS = 4    # effective batch = 4
LR          = 5e-5 # fine-tuning: lower LR than original training
VALID_SPLIT = 0.2
N_WORKERS = 0

# ── Loss weights ──────────────────────────────────────────────────────────────
LAMBDA_TD   = 1.0    # time-domain L1
LAMBDA_SPEC = 1.0    # multi-scale spectrogram L1

# STFT sizes for the multi-scale spectrogram loss (same as HTDemucs training)
STFT_SIZES  = [512, 1024, 2048, 4096]

# ── Mixed precision ───────────────────────────────────────────────────────────
USE_AMP = True   # FP16 AMP — strongly recommended to avoid CUDA OOM

# ── Misc ──────────────────────────────────────────────────────────────────────
SOURCES = ["drums", "bass", "other", "vocals"]
COLORS  = ["#4c72b0", "#55a868", "#c44e52", "#8172b2"]

def _pick_free_gpu() -> torch.device:
    """Return the CUDA device with the most free memory; fall back to CPU."""
    if not torch.cuda.is_available():
        return torch.device("cpu")
    best_idx, best_free = 0, -1
    for i in range(torch.cuda.device_count()):
        try:
            free, _ = torch.cuda.mem_get_info(i)
        except Exception:
            free = 0  # GPU fully occupied or unavailable
        if free > best_free:
            best_free, best_idx = free, i
    if best_free <= 0:
        print("Warning: all GPUs are fully occupied. Using cuda:0 anyway.")
    return torch.device(f"cuda:{best_idx}")

DEVICE  = _pick_free_gpu()

print(f"Dataset: {DATASET_ROOT}")
if training:
    print(f"Checkpoint will be saved to: {CKPT_PATH}")
else:
    print(f"Checkpoint loaded: {CKPT_PATH}")
print(f"Run    : {_run_ts}")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(DEVICE)}")
    print(f"VRAM: {torch.cuda.get_device_properties(DEVICE).total_memory / 1e9:.1f} GB")

Device : cuda:1
VRAM   : could not query — restart the kernel to release GPU memory from the previous run
Run ts : 20260407_151034
CKPT   : /nas/home/macerbi/msslnet/runs/htdemucs_finetune/htdemucs_20260407_151034.pt


## Load HTDemucs

The full pre-trained model is loaded and **all weights are unfrozen** — we are fine-tuning  
the backbone, not just spatial correction heads.

> If you want to freeze the encoder and only fine-tune the decoder, set `FREEZE_ENCODER = True`.

In [3]:
FREEZE_ENCODER = False   # True: only fine-tune the decoder (saves memory, faster)

bag  = get_model("htdemucs")
model = bag.models[0] if hasattr(bag, "models") else bag
model = model.to(DEVICE)

SAMPLE_RATE = model.samplerate
SEG_LEN     = int(float(model.segment) * SAMPLE_RATE)

if FREEZE_ENCODER:
    # Freeze all encoder / transformer layers; leave only the decoder trainable.
    # Heuristic: freeze everything whose name contains 'encoder' or 'tencoder'.
    for name, param in model.named_parameters():
        if "encoder" in name or "tencoder" in name:
            param.requires_grad_(False)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)

print(f"Model            : {type(model).__name__}")
print(f"Sources          : {model.sources}")
print(f"Sample rate      : {SAMPLE_RATE} Hz")
print(f"HTDemucs segment : {float(model.segment):.2f} s  ({SEG_LEN} samples)")
print(f"Trainable params : {n_trainable:,}")
print(f"Frozen params    : {n_frozen:,}")

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## Loss Function — Multi-Domain (Time L1 + Spectrogram L1)

HTDemucs is trained with a **multi-domain loss**:

1. **Time-domain L1** — directly penalises waveform reconstruction error.  
2. **Multi-scale spectrogram L1** — L1 on STFT magnitude at several resolutions,  
   sensitive to spectral envelope errors that are invisible to time-domain losses.

Both are averaged over the $S$ sources so the total loss is scale-independent.

In [ ]:
class HTDemucsLoss(nn.Module):
    """Multi-domain loss mirroring the HTDemucs training objective.

    Args:
        lambda_td:   weight for time-domain L1 term
        lambda_spec: weight for multi-scale spectrogram L1 term
        stft_sizes:  list of FFT sizes for the multi-scale STFT loss
    """

    def __init__(
        self,
        lambda_td: float   = 1.0,
        lambda_spec: float = 1.0,
        stft_sizes: list   = None,
    ) -> None:
        super().__init__()
        self.lambda_td   = lambda_td
        self.lambda_spec = lambda_spec
        self.stft_sizes  = stft_sizes or [512, 1024, 2048, 4096]

    def _stft_mag(self, x: torch.Tensor, n_fft: int) -> torch.Tensor:
        """Compute STFT magnitude for a (B, T) or (B*C, T) waveform."""
        hop = n_fft // 4
        win = torch.hann_window(n_fft, device=x.device)
        spec = torch.stft(
            x, n_fft=n_fft, hop_length=hop, win_length=n_fft,
            window=win, return_complex=True,
        )
        return spec.abs()  # (B*C, F, T_frames)

    def forward(
        self,
        estimates: torch.Tensor,   # (B, S, 2, T)
        targets: torch.Tensor,     # (B, S, 2, T)
    ) -> torch.Tensor:
        B, S, C, T = estimates.shape
        loss_td   = torch.tensor(0.0, device=estimates.device)
        loss_spec = torch.tensor(0.0, device=estimates.device)

        for s in range(S):
            est_s = estimates[:, s]   # (B, 2, T)
            tgt_s = targets[:, s]     # (B, 2, T)

            # Time-domain L1
            if self.lambda_td > 0:
                loss_td = loss_td + F.l1_loss(est_s, tgt_s)

            # Multi-scale spectrogram L1
            if self.lambda_spec > 0:
                # Flatten B and C into a single batch dim for stft
                est_flat = est_s.reshape(B * C, T)
                tgt_flat = tgt_s.reshape(B * C, T)
                for n_fft in self.stft_sizes:
                    mag_est = self._stft_mag(est_flat, n_fft)
                    mag_tgt = self._stft_mag(tgt_flat, n_fft)
                    loss_spec = loss_spec + F.l1_loss(mag_est, mag_tgt)

        # Normalise by number of sources (and STFT scales)
        n_scales = len(self.stft_sizes) if self.lambda_spec > 0 else 1
        total = (
            self.lambda_td   * loss_td / S
            + self.lambda_spec * loss_spec / (S * n_scales)
        )
        return total


criterion = HTDemucsLoss(
    lambda_td=LAMBDA_TD,
    lambda_spec=LAMBDA_SPEC,
    stft_sizes=STFT_SIZES,
)
print("Loss: HTDemucsLoss")
print(f"  λ_td   = {LAMBDA_TD}   (time-domain L1)")
print(f"  λ_spec = {LAMBDA_SPEC}   (multi-scale STFT L1 @ {STFT_SIZES})")

## Dataset

In [ ]:
full_ds = MusdbSpatialDataset(
    DATASET_ROOT,
    split="train",
    sources=SOURCES,
    segment_len=SEG_LEN,
    sample_rate=SAMPLE_RATE,
    augment=True,
)


n_valid  = max(1, int(len(full_ds) * VALID_SPLIT))
n_train  = len(full_ds) - n_valid
train_ds, valid_ds = random_split(full_ds, [n_train, n_valid])
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=N_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
valid_loader = DataLoader(
    valid_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=N_WORKERS, pin_memory=(DEVICE.type == "cuda"),
)
print(f"Train tracks : {n_train}   ({len(train_loader)} batches @ batch={BATCH_SIZE}, accum={ACCUM_STEPS})")
print(f"Valid tracks : {n_valid}   ({len(valid_loader)} batches)")
print(f"Effective batch size: {BATCH_SIZE * ACCUM_STEPS}")

## Training Loop

Key implementation details:

- **AMP (`autocast`)** — forward pass in FP16; scaler handles gradient scaling to prevent underflow.
- **Gradient accumulation** — `optimizer.step()` is called every `ACCUM_STEPS` batches,  
  giving an effective batch size of `BATCH_SIZE × ACCUM_STEPS` without extra VRAM.
- **Gradient clipping** (`max_norm=5`) — stabilises fine-tuning of the large backbone.

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-4,
)
scaler    = GradScaler(enabled=USE_AMP)

# Cosine annealing — one full cycle over all epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=LR / 20,
)

print(f"Optimizer : AdamW  lr={LR}  wd=1e-4")
print(f"Scheduler : CosineAnnealingLR  T_max={N_EPOCHS}  eta_min={LR/20:.2e}")
print(f"AMP       : {USE_AMP}")

In [ ]:
train_losses, valid_losses = [], []
best_valid = math.inf

@torch.no_grad()
def run_valid(model, loader):
    model.eval()
    tot, n = 0.0, 0
    for mix, targets in loader:
        mix, targets = mix.to(DEVICE), targets.to(DEVICE)
        with autocast(enabled=USE_AMP):
            estimates = model(mix)    # (B, S, 2, T)
            loss      = criterion(estimates, targets)
        tot += loss.item()
        n   += 1
    return tot / max(n, 1)


for epoch in range(1, N_EPOCHS + 1):
    model.train()
    t0         = time.time()
    tot_loss   = 0.0
    n_batches  = 0

    optimizer.zero_grad()

    for step, (mix, targets) in enumerate(train_loader):
        mix, targets = mix.to(DEVICE), targets.to(DEVICE)

        with autocast(enabled=USE_AMP):
            estimates = model(mix)                   # (B, S, 2, T)
            loss      = criterion(estimates, targets)
            # Scale loss by accumulation steps so gradients are averaged
            loss_scaled = loss / ACCUM_STEPS

        scaler.scale(loss_scaled).backward()

        tot_loss  += loss.item()
        n_batches += 1

        # Optimizer step every ACCUM_STEPS batches (or at the last batch)
        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=5.0,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    scheduler.step()

    train_loss = tot_loss / max(n_batches, 1)
    valid_loss = run_valid(model, valid_loader) if valid_loader.dataset else float("nan")

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    elapsed = time.time() - t0
    print(
        f"Epoch {epoch:4d}/{N_EPOCHS} "
        f"train={train_loss:.4f}  "
        f"valid={valid_loss:.4f}  "
        f"lr={scheduler.get_last_lr()[0]:.2e}  "
        f"({elapsed:.0f}s)"
    )

    # Save best checkpoint
    if valid_loss < best_valid:
        best_valid = valid_loss
        torch.save(
            {
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "optim_state": optimizer.state_dict(),
                "train_loss":  train_loss,
                "valid_loss":  valid_loss,
            },
            CKPT_PATH,
        )
        print(f"           ↳ checkpoint saved (valid={valid_loss:.4f})")

print("\nTraining complete.")
print(f"Best validation loss : {best_valid:.4f}")
print(f"Checkpoint           : {CKPT_PATH}")

## Training Curves

In [ ]:
epochs = range(1, len(train_losses) + 1)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, train_losses, label="train",  color="#4c72b0", linewidth=1.5)
ax.plot(epochs, valid_losses, label="valid",  color="#c44e52", linewidth=1.5, linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (td L1 + spec L1)")
ax.set_title("HTDemucs fine-tuning — binauralMUSDB18")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Load Checkpoint and Evaluate SDR

Reload the best checkpoint and compute the **Signal-to-Distortion Ratio (SDR)**  
on the test split using `museval` / `mir_eval`.

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt['epoch']}  valid_loss={ckpt['valid_loss']:.4f}")

In [ ]:
test_ds = MusdbSpatialDataset(
    DATASET_ROOT,
    split="test",
    sources=SOURCES,
    segment_len=SEG_LEN,
    sample_rate=SAMPLE_RATE,
    augment=False,
)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

sdr_per_source = {src: [] for src in SOURCES}

with torch.no_grad():
    for mix, targets in test_loader:
        mix, targets = mix.to(DEVICE), targets.to(DEVICE)
        with autocast(enabled=USE_AMP):
            estimates = model(mix)   # (1, S, 2, T)

        for s, src in enumerate(SOURCES):
            est = estimates[0, s].cpu().numpy()   # (2, T)
            tgt = targets[0, s].cpu().numpy()     # (2, T)
            # Simple waveform SDR: 10*log10(||target||^2 / ||target - estimate||^2)
            noise_power = np.sum((tgt - est) ** 2) + 1e-8
            sig_power   = np.sum(tgt ** 2) + 1e-8
            sdr = 10 * np.log10(sig_power / noise_power)
            sdr_per_source[src].append(sdr)

print("\nTest SDR (waveform, dB) — segment-level average:")
print(f"  {'Source':<10} {'Mean SDR':>10}  {'Std':>8}")
print("  " + "-" * 32)
all_sdrs = []
for src in SOURCES:
    vals = sdr_per_source[src]
    print(f"  {src:<10} {np.mean(vals):>10.2f}  {np.std(vals):>8.2f}")
    all_sdrs.extend(vals)
print("  " + "-" * 32)
print(f"  {'Average':<10} {np.mean(all_sdrs):>10.2f}  {np.std(all_sdrs):>8.2f}")